# Natural-language → geometry, end to end

## What this shows

Three new surfaces, wired together:

* **Gazetteer protocol** (`siege_utilities.geo.gazetteers`) — name → geometry resolution. WKLS is the default global backend (Overture admin boundaries; no API key); Nominatim wraps geopy with `geometry='geojson'`.
* **Etter parser** (`siege_utilities.geo.providers.etter_filter`) — LLM-driven NL → `EtterFilter` (spatial relation, reference location, buffer distance).
* **`etter_to_geometry` resolver** (`siege_utilities.geo.providers.etter_to_geometry`) — combines the two and emits a shapely geometry with three relation-semantics modes.

Together: "voters near Lausanne" → polygon you can intersect with a precinct file.

All gazetteer calls are mocked so the notebook runs offline. The real pipeline calls WKLS' embedded sedonadb / Nominatim's HTTP API on first use.

## 1. Pick a gazetteer

`resolve_gazetteer()` auto-selects the best available backend: WKLS first (global, no rate limit), Nominatim second (rate-limited public OSM). The protocol is the same either way — `lookup(name) → GazetteerResult`.

In [ ]:
from siege_utilities.geo.gazetteers import resolve_gazetteer, Gazetteer

# For the notebook we build a mock that resolves to a known shape;
# real usage is just `gz = resolve_gazetteer()`.
from shapely.geometry import Polygon
from siege_utilities.geo.gazetteers.base import GazetteerResult

_LAUSANNE = Polygon([
    (6.55, 46.45), (6.65, 46.45), (6.65, 46.55), (6.55, 46.55), (6.55, 46.45),
])

class _DemoGazetteer:
    provider_name = 'demo'
    def lookup(self, name, *, country_hint=None, admin_hint=None):
        return GazetteerResult(
            name=name, canonical_path=('CH', name),
            geometry=_LAUSANNE, centroid=_LAUSANNE.centroid,
            country='CH', source='demo',
        )
    def search(self, name, **kwargs): return []
    def is_available(self): return True

gz = _DemoGazetteer()
isinstance(gz, Gazetteer)  # the protocol is structural — no inheritance needed

## 2. Parse natural language with Etter

In production, `EtterParser` calls an LLM. For the notebook we fabricate an `EtterFilter` directly — it's the same shape the parser produces.

In [ ]:
from siege_utilities.geo.providers.etter_filter import EtterFilter

filter_ = EtterFilter(
    original_query='voters near Lausanne within 5 km',
    spatial_relation='near',
    reference_location='Lausanne',
    buffer_distance_m=5000.0,
    confidence=0.94,
)
filter_

## 3. Resolve to a geometry

The default `RelationSemantics.BOUNDED` mode is what you want for indexed-lookup workloads — the output is always a finite polygon you can intersect with a precinct table or hand to a spatial join.

In [ ]:
from siege_utilities.geo.providers.etter_to_geometry import (
    etter_to_geometry, RelationSemantics,
)

result = etter_to_geometry(filter_, gazetteer=gz)
print('relation        :', result.relation)
print('buffer used (km):', result.buffer_km)
print('semantics       :', result.semantics.value)
print('geometry type   :', result.geometry.geom_type)
print('bounds          :', result.geometry.bounds)
print('notes           :', result.notes)

## 4. Directional relations: three flavours

"North of Lake Geneva" can mean three different things depending on what you want to do with the result. The resolver lets you pick.

In [ ]:
directional = EtterFilter(
    original_query='cantons north of Lake Geneva',
    spatial_relation='north_of',
    reference_location='Lake Geneva',
    confidence=0.91,
)

bounded = etter_to_geometry(directional, gazetteer=gz, default_buffer_km=20)
halfplane = etter_to_geometry(directional, gazetteer=gz, semantics=RelationSemantics.HALFPLANE)
predicate = etter_to_geometry(directional, gazetteer=gz, semantics=RelationSemantics.CONTAINS_CENTROID)

print('BOUNDED   bounds :', tuple(round(x, 2) for x in bounded.geometry.bounds))
print('HALFPLANE bounds :', tuple(round(x, 2) for x in halfplane.geometry.bounds))
print('CONTAINS_CENTROID geometry type:', type(predicate.geometry).__name__)

from shapely.geometry import Point
north_point = Point(6.6, 50.0)  # ~Strasbourg latitude
south_point = Point(6.6, 40.0)  # ~Rome latitude
print('predicate(north Strasbourg-ish):', predicate.geometry(north_point))
print('predicate(south Rome-ish)      :', predicate.geometry(south_point))

## 5. Hand off to the rest of the toolkit

The geometry the resolver returns is just a shapely shape. From here it's ordinary geopandas:

* feed it to a `BoundaryProvider` to clip TIGER blocks to the region,
* drop it into the multi-engine `spatial_join` to filter a voter file,
* or render it as a hex cartogram against an aggregated metric.

In [ ]:
# Example: build a tiny GeoDataFrame around the BOUNDED result and use
# the H3 / S2 grid dispatcher to index it.
import geopandas as gpd
from siege_utilities.geo.grids import index_polygon

gdf = gpd.GeoDataFrame({'name': ['near Lausanne']}, geometry=[bounded.geometry], crs='EPSG:4326')
cells_h3 = index_polygon(gdf.geometry.iloc[0], grid='h3', resolution=5)
print(f'{len(cells_h3)} H3 cells at resolution 5 cover the bounded buffer')

## Why three semantics modes

| Mode | Output | Best for |
|---|---|---|
| `BOUNDED` (default) | Finite polygon | Spatial joins, indexed lookups, choropleth clipping |
| `HALFPLANE` | Unbounded polygon (clipped to world bbox) | Filter semantics where you'll intersect with a country/region later |
| `CONTAINS_CENTROID` | `PointPredicate` callable | Per-candidate filter; no polygon materialised |

Pick the one that matches what the caller will do next. The default is `BOUNDED` because it's the one that composes with everything else.

## Where this fits in the larger pipeline

```mermaid
graph LR
  NL[NL query] --> Etter[EtterParser]
  Etter --> Filter[EtterFilter]
  Filter --> Resolver[etter_to_geometry]
  Gaz[Gazetteer<br/>WKLS / Nominatim] --> Resolver
  Resolver --> Geom[shapely geometry]
  Geom --> SpatialJoin[engine.spatial_join]
  Geom --> Grid[index_polygon]
  Geom --> Hex[hex_tile_map]
```